# Practical Exercise 4 — Knowledge-Based Trailer Agent

### Step 1 — Identify the task

We are building an agent to help a self-driving trailer decide which town to head to next on its way from Nairobi to Kisumu.

**Goal:** Reach Kisumu using the shortest available route in our road network. If a road is reported closed, the agent should find another route. If no route is available, it should tell the trailer to wait.

### Step 2 — Assemble relevant knowledge

We’re using the highway network from Practical Exercise 3. It shows which towns the trailer can travel between, the direction of each road and the distance in kilometres. We’ll use those connections to build the search tree and compare the possible routes from Nairobi to Kisumu.

The agent also needs to know the trailer’s current town, its destination and which roads have been reported closed. We’ll enter that information ourselves when testing the agent.




### Step 3 — Define the vocabulary

**Predicates**

- `Road(A, B, d)` means our graph has a road from town A to town B that is d kilometres long.
- `ClosedRoad(A, B)` means the road from A to B has been reported closed.
- `Traversable(A, B, d)` means the agent can use that road. In our model, a road is available unless we report it closed.
- `At(T)` means the trailer is currently in town T.
- `Goal(T)` means town T is the trailer’s destination.

**Constants**

- Town names like `Nairobi`, `Maai Mahiu`, `Narok` and `Kisumu`.
- Road report values `open` and `closed`.

For example, `Road(Nairobi, Maai Mahiu, 55)` means the road from Nairobi to Maai Mahiu is 55 km in our graph. If we add `ClosedRoad(Nairobi, Maai Mahiu)`, the agent skips that road when planning the route.

### Step 4 — Encode general knowledge about the domain

We store the roads and distances from Exercise 3 in a Python dictionary. Each town lists the towns the trailer can go to next and how far away they are.

When planning a route, the agent leaves out any road that has been reported closed. It then uses Uniform Cost Search to find the shortest route through the remaining roads. If it finds a route, it recommends the first town on that route. If it can’t find one, it tells the trailer to wait. If the trailer is already at its destination, it reports that it has arrived.

In [2]:
graph = {
    "Nairobi": {"Thika": 45, "Maai Mahiu": 55, "Narok": 145},
    "Thika": {"Murang'a": 47},
    "Murang'a": {"Nyeri": 65},
    "Nyeri": {"Nyahururu": 104},
    "Nyahururu": {"Nakuru": 67},
    "Maai Mahiu": {"Naivasha": 34},
    "Naivasha": {"Gilgil": 30},
    "Gilgil": {"Nakuru": 42},
    "Nakuru": {"Molo": 47},
    "Molo": {"Londiani": 22},
    "Londiani": {"Kericho": 49},
    "Narok": {"Bomet": 81},
    "Bomet": {"Kericho": 76},
    "Kericho": {"Kisumu": 84},
    "Kisumu": {},
}

print("Towns:", len(graph))
print("Directed roads:", sum(len(roads) for roads in graph.values()))

Towns: 15
Directed roads: 16


In [3]:
import heapq

def uniform_cost_search(roads, start, goal):
    queue = [(0, start, [start])]
    best_cost = {start: 0}

    while queue:
        cost, town, path = heapq.heappop(queue)

        if cost > best_cost[town]:
            continue

        if town == goal:
            return path, cost

        for next_town, distance in roads[town].items():
            new_cost = cost + distance
            if new_cost < best_cost.get(next_town, float("inf")):
                best_cost[next_town] = new_cost
                heapq.heappush(queue, (new_cost, next_town, path + [next_town]))

    return None, None


def trailer_decision(current_town, destination, closed_roads):
    if current_town not in graph or destination not in graph:
        return "Enter a town from the road network.", None, None

    if current_town == destination:
        return "ARRIVED", [current_town], 0

    # Keep only the roads the trailer can use.
    available_roads = {
        town: {
            next_town: distance
            for next_town, distance in roads.items()
            if (town, next_town) not in closed_roads
        }
        for town, roads in graph.items()
    }

    path, distance = uniform_cost_search(
        available_roads, current_town, destination
    )

    if path is None:
        return "WAIT", None, None

    return f"DRIVE_TO {path[1]}", path, distance

### Step 5 — Add new facts to the knowledge base

We’ll report that the road from Nairobi to Maai Mahiu is closed. We store the closure as a pair of towns, with Nairobi first because that’s the direction the report applies to. The next time the agent plans a route, it will skip that road.

In [4]:
closed_roads = set()
town_names = {town.casefold(): town for town in graph}


def add_road_report(origin, destination, status):
    origin = town_names.get(origin.strip().casefold())
    destination = town_names.get(destination.strip().casefold())
    status = status.strip().casefold()

    if origin is None or destination is None:
        print("Enter town names from the road network.")
        return

    if destination not in graph[origin]:
        print("That road is not in the graph in that direction.")
        return

    road = (origin, destination)

    if status == "closed":
        closed_roads.add(road)
    elif status == "open":
        closed_roads.discard(road)
    else:
        print("Use 'open' or 'closed' for the road status.")
        return

    print(f"Road report saved: {origin} → {destination} ({status})")


# Add a sample closure directly to the knowledge base.
add_road_report("Nairobi", "Maai Mahiu", "closed")

# Let the user enter another road report.
if input("Add another road report? (yes/no): ").strip().casefold() == "yes":
    origin = input("Town the road starts from: ")
    road_end = input("Town the road leads to: ")
    status = input("Road status (open/closed): ")
    add_road_report(origin, road_end, status)

print("\nRoads reported closed:")
if closed_roads:
    for origin, road_end in sorted(closed_roads):
        print(f"- {origin} → {road_end}")
else:
    print("None")

Road report saved: Nairobi → Maai Mahiu (closed)
Add another road report? (yes/no): yes
Town the road starts from: Narok
Town the road leads to: Bomet
Road status (open/closed): closed
Road report saved: Narok → Bomet (closed)

Roads reported closed:
- Nairobi → Maai Mahiu
- Narok → Bomet


### Step 6 — Encode the specific problem instance

For this test, the trailer starts in Nairobi and needs to reach Kisumu. We enter the current town and destination when we run the cell. The agent will use these details together with the road reports from Step 5.

In [8]:
def read_town(prompt, default):
    while True:
        answer = input(prompt).strip() or default
        town = town_names.get(answer.casefold())

        if town is not None:
            return town

        print("Enter a town from the road network.")


current_town = read_town("Current town [Nairobi]: ", "Nairobi")
destination = read_town("Destination [Kisumu]: ", "Kisumu")

print(f"Trip: {current_town} → {destination}")

Current town [Nairobi]: 
Destination [Kisumu]: 
Trip: Nairobi → Kisumu


### Step 7 — Pose a query and get an answer

We ask the agent which town the trailer should head to next. It uses the current town and destination from Step 6 and skips the roads we reported closed in Step 5.

In [9]:
action, path, distance = trailer_decision(
    current_town, destination, closed_roads
)

print("Current town:", current_town)
print("Destination:", destination)
print("Agent decision:", action)

if path is not None:
    print("Route:", " → ".join(path))
    print("Total distance:", distance, "km")

Current town: Nairobi
Destination: Kisumu
Agent decision: DRIVE_TO Thika
Route: Nairobi → Thika → Murang'a → Nyeri → Nyahururu → Nakuru → Molo → Londiani → Kericho → Kisumu
Total distance: 530 km


### Step 8 — Query the knowledge base

We asked the agent about three trips using the same road reports. We wanted to see what it would recommend if the trailer could still reach Kisumu, if it was already there, or if it had no available route.

In [10]:
queries = [
    ("Nakuru", "Kisumu"),
    ("Kisumu", "Kisumu"),
    ("Narok", "Kisumu"),
]

for start, goal in queries:
    action, route, distance = trailer_decision(start, goal, closed_roads)

    print(f"\nTrip: {start} → {goal}")
    print("Agent decision:", action)

    if route is not None:
        print("Route:", " → ".join(route))
        print("Total distance:", distance, "km")


Trip: Nakuru → Kisumu
Agent decision: DRIVE_TO Molo
Route: Nakuru → Molo → Londiani → Kericho → Kisumu
Total distance: 202 km

Trip: Kisumu → Kisumu
Agent decision: ARRIVED
Route: Kisumu
Total distance: 0 km

Trip: Narok → Kisumu
Agent decision: WAIT


### Step 9 — Debug the knowledge base

We tested what happens when someone enters a town outside our road network or reports a road in the wrong direction. We also checked that reopening Nairobi → Maai Mahiu brings back the original 363 km route.

In [5]:
# Check that an unknown town is handled.
action, _, _ = trailer_decision("Unknown Town", "Kisumu", closed_roads)
print("Unknown town:", action)

# Check that a report in the wrong direction is rejected.
print("\nTesting Bomet → Narok:")
original_closures = closed_roads.copy()
add_road_report("Bomet", "Narok", "closed")
assert closed_roads == original_closures
print("Road reports stayed the same.")

# Test reopening a road on a copy so Step 8 keeps its closures.
test_closures = closed_roads.copy()
test_closures.discard(("Nairobi", "Maai Mahiu"))

action, route, distance = trailer_decision("Nairobi", "Kisumu", test_closures)
assert action == "DRIVE_TO Maai Mahiu" and distance == 363

print("\nAfter reopening Nairobi → Maai Mahiu:")
print("Agent decision:", action)
print("Total distance:", distance, "km")

Unknown town: Enter a town from the road network.

Testing Bomet → Narok:
That road is not in the graph in that direction.
Road reports stayed the same.

After reopening Nairobi → Maai Mahiu:
Agent decision: DRIVE_TO Maai Mahiu
Total distance: 363 km


### Step 10 — Final reflection

When we closed Nairobi → Maai Mahiu and Narok → Bomet, the agent chose the route through Thika, which was 530 km. When we reopened Nairobi → Maai Mahiu in our test, it went back to the original 363 km route. We also checked that it rejected a closure report for Bomet → Narok because our graph only has that connection in the other direction.

One limitation is that we enter road closures by hand. If nobody reports a closure, the agent assumes the road is open. In a future version, we could use live road reports and record when each report was last updated.